In [ ]:
!pip install owlready2

In [ ]:
#imports
import nltk
from nltk import word_tokenize, pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from collections import defaultdict, Counter

import pandas as pd

from owlready2 import *
import os

from openai import OpenAI

In [ ]:
df= pd.read_json('/content/test_data_subtask_1.json')
all_sentences = df_test['syllogism'].values.tolist()
all_sentences = '; '.join(all_sentences)

In [ ]:
predictions = []
counter = 0
for i in range(len(df)):
  syllogism = df.iloc[i]['syllogism']
  counter += 1
  prompt = f"""
  You are an expert in knowledge representation, Description Logics, and the Semantic Web.
  Your task is to convert a syllogism expressed in natural language into an OWL TBox schema using RDF/XML
  Input characteristics

    The input consists of two premises and a conclusion.

  Core principle

    Do NOT attempt to resolve, correct, or normalize logical inconsistencies.
    Always preserve the polarity (affirmative or negative) exactly as stated in each sentence, including the conclusion.
    Maintain the categories and properties as stated.
    Change only conectors, verbs and adverbs which relate to logical conditions into logical relations existent in OWL.

  Mapping rules

    Common nouns denoting kinds → OWL Classes (singular, capitalized).
    Specific entities (e.g., “el león”) → model as OWL Classes (not individuals).
    Verbs or verbal predicates → OWL Object Properties (lowercase, infinitive).

  Output constraints

    Output only RDF/XML usable via owlready2.
    No explanations, comments, or extra text.
    Keep class and property names consistent across all axioms.

  Goal

    The output ontology may be logically inconsistent and must reflect this faithfully.
    The ontology should be suitable for automated inconsistency detection with a DL reasoner.

  Example:

      <?xml version="1.0"?>
      <rdf:RDF xmlns="http://www.semanticweb.org/fbobillo/ontologies/silogismo#"
          xml:base="http://www.semanticweb.org/fbobillo/ontologies/silogismo"
          xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
          xmlns:owl="http://www.w3.org/2002/07/owl#"
          xmlns:xml="http://www.w3.org/XML/1998/namespace"
          xmlns:xsd="http://www.w3.org/2001/XMLSchema#"
          xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#">
          <owl:Ontology rdf:about="http://www.semanticweb.org/fbobillo/ontologies/silogismo"/>


          <!-- http://www.semanticweb.org/fbobillo/ontologies/silogismo#respira -->

          <owl:ObjectProperty rdf:about="http://www.semanticweb.org/fbobillo/ontologies/silogismo#respira"/>



          <!-- http://www.semanticweb.org/fbobillo/ontologies/silogismo#Animal -->

          <owl:Class rdf:about="http://www.semanticweb.org/fbobillo/ontologies/silogismo#Animal">
              <rdfs:subClassOf>
                  <owl:Restriction>
                      <owl:onProperty rdf:resource="http://www.semanticweb.org/fbobillo/ontologies/silogismo#respira"/>
                      <owl:someValuesFrom rdf:resource="http://www.w3.org/2002/07/owl#Thing"/>
                  </owl:Restriction>
              </rdfs:subClassOf>
          </owl:Class>



          <!-- http://www.semanticweb.org/fbobillo/ontologies/silogismo#Leon -->

          <owl:Class rdf:about="http://www.semanticweb.org/fbobillo/ontologies/silogismo#Leon">
              <rdfs:subClassOf rdf:resource="http://www.semanticweb.org/fbobillo/ontologies/silogismo#Animal"/>
              <rdfs:subClassOf>
                  <owl:Class>
                      <owl:complementOf>
                          <owl:Restriction>
                              <owl:onProperty rdf:resource="http://www.semanticweb.org/fbobillo/ontologies/silogismo#respira"/>
                              <owl:someValuesFrom rdf:resource="http://www.w3.org/2002/07/owl#Thing"/>
                          </owl:Restriction>
                      </owl:complementOf>
                  </owl:Class>
              </rdfs:subClassOf>
          </owl:Class>



          <!-- http://www.w3.org/2002/07/owl#Thing -->

          <owl:Class rdf:about="http://www.w3.org/2002/07/owl#Thing"/>
      </rdf:RDF>

  Syllogism to solve:
    {syllogism}

  """

  client = OpenAI(
    api_key= key
  )


  response = client.responses.create(
                              model="gpt-5-nano", #gpt-5-nano
                              input= prompt
                              )

  response_clean = response.output_text
  lines = response_clean.splitlines()

  # Remove first and last lines
  trimmed_response = "\n".join(lines[1:-1])


  FILENAME = f"/content/silogismo_{counter}.ttl"

  with open(FILENAME, "w", encoding="utf-8") as f:
    f.write(trimmed_response)

  try:

    # Cargar la ontología (no soporta Manchester syntax)
    FILENAME = f"/content/silogismo_{counter}.ttl"
    world = World()
    onto_path.append(os.getcwd())
    try:
      onto = world.get_ontology(FILENAME).load()
      # Usar razonador
      try:
          #print("\nEjecutando razonador...")
          # Usa HermiT si Java está instalado, si no, Pellet-lite
          sync_reasoner(world, debug=False)
          num = len(list(world.inconsistent_classes()))
          #if num > 1:
            #print("Hay clases inconsistentes")
          predictions.append(num)
      except OwlReadyInconsistentOntologyError:
          #print("\nOntología inconsistente.")
          predictions.append(None)
    except OwlReadyOntologyParsingError:
      predictions.append(None)
      print(counter)
  except:
    predictions.append(None)
    print(counter)

In [ ]:
df['predictions'] = predictions

In [ ]:
df['validity'] = df['predictions'].apply(lambda x: True if x > 0 else False)
df = df[['id', 'validity']]

In [ ]:
import json
df_list = df.to_dict(orient='records')
with open('predictions.json', 'w') as f:
    json.dump(df_list, f, indent=2)